In [70]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=r"C:\E drive\A.I Classes\Agentic and Gen AI with Cloud\.env")
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [71]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")
result=llm.invoke("Hey")
print(result.content)

Hey there! 👋 How can I help you today?


In [72]:
from langchain_community.document_loaders import WebBaseLoader

url=["https://reference.langchain.com/python/langchain-community/document_loaders/web_base/WebBaseLoader",
     "https://reference.langchain.com/python/langchain/overview"]

loader=WebBaseLoader(url)
doc=loader.load()


In [73]:
doc

[Document(metadata={'source': 'https://reference.langchain.com/python/langchain-community/document_loaders/web_base/WebBaseLoader', 'title': 'WebBaseLoader | langchain_community | LangChain Reference', 'description': 'Python API reference for document_loaders.web_base.WebBaseLoader in langchain_community. Part of the LangChain ecosystem.', 'language': 'en'}, page_content='WebBaseLoader | langchain_community | LangChain ReferenceLangChain Reference home pageSearch...⌘KAsk AIGitHubMain DocsDeep AgentsLangChainLangGraphIntegrationsLangSmithOverviewAmazon NovaAnthropicAstraDBAWSAzure (Microsoft)CerebrasChromaCohereCommunityOverviewChat ModelsLLMsEmbeddingsVector StoresDocument LoadersTools & AgentsRetrieversCachesChat HistoryGraphsUtilitiesDb2DeepSeekElasticsearchExaFireworksGoogle (Community)Google GenAI (Gemini)Google Vertex AIGroqHuggingFaceIBMLiteLLMMilvusMistral AINeo4JNomicNvidia AI EndpointsOllamaOpenAIOpenRouterParallelPerplexityPineconePostgresQdrantRedisSema4SnowflakeSQLServerTav

In [74]:
doc[0].metadata['title']


'WebBaseLoader | langchain_community | LangChain Reference'

In [75]:
for idx, dc in enumerate(doc):
    print(f"Document {idx+1}:--")
    print(f"The tile of Document is {dc.metadata['title']}")
    print(f"len of document is {len(dc.page_content)}")
    print(f"The description of Document is {dc.metadata['description']}")
    print(f"The description of Document is {dc.page_content[:500]}")
    print(f"----"*150)

Document 1:--
The tile of Document is WebBaseLoader | langchain_community | LangChain Reference
len of document is 6725
The description of Document is Python API reference for document_loaders.web_base.WebBaseLoader in langchain_community. Part of the LangChain ecosystem.
The description of Document is WebBaseLoader | langchain_community | LangChain ReferenceLangChain Reference home pageSearch...⌘KAsk AIGitHubMain DocsDeep AgentsLangChainLangGraphIntegrationsLangSmithOverviewAmazon NovaAnthropicAstraDBAWSAzure (Microsoft)CerebrasChromaCohereCommunityOverviewChat ModelsLLMsEmbeddingsVector StoresDocument LoadersTools & AgentsRetrieversCachesChat HistoryGraphsUtilitiesDb2DeepSeekElasticsearchExaFireworksGoogle (Community)Google GenAI (Gemini)Google Vertex AIGroqHuggingFaceIBMLiteLLMMilvusMistral
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [76]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
chunks=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
splitter=chunks.split_documents(dc for dc in doc)
print(f"The number of chunks are : {len(splitter)}")

The number of chunks are : 23


In [77]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [81]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(documents=splitter,embedding=embeddings)

retriever=vectorstore.as_retriever()
retriever.invoke("What is WebBaseLoader?")


[Document(id='f7809333-a1fd-4dc0-bf7b-64ea476e82cf', metadata={'source': 'https://reference.langchain.com/python/langchain-community/document_loaders/web_base/WebBaseLoader', 'title': 'WebBaseLoader | langchain_community | LangChain Reference', 'description': 'Python API reference for document_loaders.web_base.WebBaseLoader in langchain_community. Part of the LangChain ecosystem.', 'language': 'en'}, page_content='WebBaseLoader | langchain_community | LangChain ReferenceLangChain Reference home pageSearch...⌘KAsk AIGitHubMain DocsDeep AgentsLangChainLangGraphIntegrationsLangSmithOverviewAmazon NovaAnthropicAstraDBAWSAzure (Microsoft)CerebrasChromaCohereCommunityOverviewChat ModelsLLMsEmbeddingsVector StoresDocument LoadersTools & AgentsRetrieversCachesChat HistoryGraphsUtilitiesDb2DeepSeekElasticsearchExaFireworksGoogle (Community)Google GenAI (Gemini)Google Vertex AIGroqHuggingFaceIBMLiteLLMMilvusMistral'),
 Document(id='d5d7cc47-9248-4251-a5c5-cb7459f56acb', metadata={'source': 'http

In [82]:
from langchain_core.tools import create_retriever_tool

retriever_tool=create_retriever_tool(
    retriever,
    "retriever_vector_db_blog",
    "Search and run information about Langgraph"
)

In [84]:
retriever_tool

StructuredTool(name='retriever_vector_db_blog', description='Search and run information about Langgraph', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000025D96A20E00>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000025D96A21800>)

In [83]:
tools=[retriever_tool]

In [85]:
llm=llm.bind_tools(tools)

In [ ]:
from pydantic import BaseModel
from typing_extensions import TypedDict
from typing import Annotated, Sequence
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage, BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.messages import PromptTemplate

ModuleNotFoundError: No module named 'langchain_core.message'

In [95]:

class web(BaseModel):
    messages:Annotated[Sequence[AnyMessage],add_messages]

In [97]:
def agent(state):
    """
    Invokes the agent model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply end.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with the agent response appended to messages
    """
    print("---CALL AGENT---")
    messages = state["messages"]
    model = ChatGroq(model="openai/gpt-oss-20b")
    model = model.bind_tools(tools)
    response = model.invoke(messages)
    return {"messages": [response]}

In [ ]:
### Edges
def grade_documents(state) -> Literal["generate", "rewrite"]:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (messages): The current state

    Returns:
        str: A decision for whether the documents are relevant or not
    """

    print("---CHECK RELEVANCE---")

    # Data model
   
   class grade(BaseModel):
           """Binary score for relevance check."""
   
           binary_score: str = Field(description="Relevance score 'yes' or 'no'")
           
    model = ChatGroq(model="openai/gpt-oss-20b")
    model=model.with_structured_output(grade)
    
    
    prompt = PromptTemplate(
            template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
            Here is the retrieved document: \n\n {context} \n\n
            Here is the user question: {question} \n
            If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
            Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.""",
            input_variables=["context", "question"],
        )
    respond=model.invoke()
